In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")
from utils.scaling import fit_scalers, apply_scaler
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_beus_articulatory_features

In [10]:
df = pd.read_csv("../dataset_construction/datasets/processed_beus.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [11]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,ac_f,art_f,subject
0,1.47,1.53,1.53,1.60,p,l,p,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
1,1.53,1.60,1.60,1.75,l,i,p_l,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
2,1.60,1.75,1.75,1.83,i,z,p_l_i,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
3,1.83,1.97,1.97,2.16,k,ɔ,k,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
4,1.97,2.16,2.16,2.17,ɔ,l,k_ɔ,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
...,...,...,...,...,...,...,...,...,...,...,...,...
163314,2.23,2.32,2.32,2.39,h,i,h,he,6,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163315,2.39,2.53,2.53,2.67,f,ɛ,f,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163316,2.53,2.67,2.67,2.73,ɛ,l,f_ɛ,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163317,2.67,2.73,2.73,2.79,l,i,f_ɛ_l,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import VideoAutoencoder, SequenceAutoencoder

device = "cpu"

art_model = VideoAutoencoder(
    input_hw=(56, 112),
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
    input_channels=2,
    output_activation=None,
    stn_mode="separate",
    stn_transform="rigid"

).to(device)
ckpt = torch.load(
    "../../encoder_training/ckpts/beus/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/beus/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [13]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,ac_f,art_f,subject
0,1.47,1.53,1.53,1.60,p,l,p,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
1,1.53,1.60,1.60,1.75,l,i,p_l,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
2,1.60,1.75,1.75,1.83,i,z,p_l_i,please,0,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
3,1.83,1.97,1.97,2.16,k,ɔ,k,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
4,1.97,2.16,2.16,2.17,ɔ,l,k_ɔ,call,1,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,00me
...,...,...,...,...,...,...,...,...,...,...,...,...
163314,2.23,2.32,2.32,2.39,h,i,h,he,6,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163315,2.39,2.53,2.53,2.67,f,ɛ,f,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163316,2.53,2.67,2.67,2.73,ɛ,l,f_ɛ,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me
163317,2.67,2.73,2.73,2.79,l,i,f_ɛ_l,fairly,7,/Volumes/Seagate Backup Plus Drive/ultrasound_...,/Volumes/Seagate Backup Plus Drive/ultrasound_...,80me


In [14]:
import cv2
with open("../../encoder_training/scalers/beus_video_stats.pkl", "rb") as f:
    _REC_STATS = pickle.load(f)

def recording_stats(path, n_sample=400):
    """
    Pixel mean and sd over the WHOLE recording, cached per file.
 
    Note cv2's CAP_PROP_POS_FRAMES seeking on compressed video snaps to the
    nearest keyframe on some codecs. That is harmless for statistics, but it is
    worth checking separately that _read_video_segment returns frames that really
    correspond to the requested times — a silent half-second offset would
    misalign every token against the audio.
    """
    if path in _REC_STATS:
        return _REC_STATS[path]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.unique(np.linspace(0, total - 1, min(n_sample, total)).astype(int))
    vals = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            vals.append(fr[:, :, 0].astype(np.float32))
    cap.release()
    if not vals:
        return None
    v = np.stack(vals)
    _REC_STATS[path] = (float(v.mean()), float(v.std()) + 1e-5)
    return _REC_STATS[path]

with open(f"../../encoder_training/scalers/beus_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [15]:
def get_vectors(art_f, ac_f, start, end, subject):
    stats = recording_stats(art_f)
    articulatory_features = get_beus_articulatory_features(
        art_f,
        start,
        end,
        stats=stats
    )

    if articulatory_features is not None:

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .to(device)
        )
        articulatory_features = articulatory_features.permute(0, 2, 1, 3, 4).contiguous()

        art_out, art_z = art_model(
            articulatory_features,
            return_latent=True
        )
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = apply_scaler(mfccs, ac_scalers[subject])
        mfccs = (
            torch.from_numpy(mfccs)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs, return_latent=True)
    else:
        ac_out, ac_z = None, None

    return {
        "articulatory_features": (
            articulatory_features.detach().cpu().numpy()
            if articulatory_features is not None
            else None
        ),
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "mfccs": (
            mfccs.detach().cpu().numpy()
            if mfccs is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [16]:
def distance(v1, v2):
    if v1 is None or v2 is None:
        return None
    return np.linalg.norm(v1 - v2)

In [17]:
pbar = tqdm.tqdm(df.iterrows(), total=len(df))

dists = {
    "art_latent_dist": [],
    # "art_raw_dist": [],
    "ac_latent_dist": [],
    # "ac_raw_dist": [],
}
for idx, data in pbar:
    
    phone1_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone1_start"], data["phone1_end"], data["subject"])
    phone2_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone2_start"], data["phone2_end"], data["subject"])

    art_latent_dist = distance(phone1_vectors["art_z"], phone2_vectors["art_z"])
    # art_raw_dist = distance(phone1_vectors["articulatory_features"], phone2_vectors["articulatory_features"])

    ac_latent_dist = distance(phone1_vectors["ac_z"], phone2_vectors["ac_z"])
    # ac_raw_dist = distance(phone1_vectors["mfccs"], phone2_vectors["mfccs"])

    dists["art_latent_dist"].append(art_latent_dist)
    # dists["art_raw_dist"].append(art_raw_dist)
    dists["ac_latent_dist"].append(ac_latent_dist)
    # dists["ac_raw_dist"].append(ac_raw_dist)

pd.DataFrame(dists).to_csv("distances_beus.csv", index=False)

100%|██████████| 163319/163319 [5:24:57<00:00,  8.38it/s]  
